# 03 - Residual Stream Interventions

Runs the core experimental conditions:
- **self_prefill**: Re-tokenize COT, fresh forward pass, sample answer (sanity check)
- **zeroed_layer_k**: Replace residual at last position with raw embedding at layer k

**IMPORTANT**: Restart the kernel before running this notebook to free GPU memory from vLLM (notebook 02).

**Set `CONDITION` in the cell below** to control which condition runs.

**Fully resumable** - caches one JSON file per (condition, problem) in `cache/interventions/`.

In [1]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/10-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/JackHopkins/legibility.git", str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, INTERVENTION_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Already up to date.


In [ ]:
# Verify GPU is free and purge errored cache files
import torch, json

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
    if free < 50e9:
        print("WARNING: Less than 50GB free. Restart the kernel to free vLLM memory.")

# Purge cached files that contain errors (so they get retried)
purged = 0
for p in INTERVENTION_CACHE.glob("*.json"):
    try:
        entry = json.loads(p.read_text())
        if entry.get("error") is not None:
            p.unlink()
            purged += 1
    except Exception:
        p.unlink()
        purged += 1

print(f"Purged {purged} errored cache files")

In [2]:
# =============================================
# SET THE CONDITION TO RUN HERE
# =============================================
# Options:
#   "self_prefill"
#   "zeroed_layer_0"
#   "zeroed_layer_9"
#   "zeroed_layer_18"
#   "zeroed_layer_27"
#   "zeroed_layer_35"
#
# Or set to "all" to run all conditions sequentially.

CONDITION = "all"

In [3]:
import json
import traceback
import torch
from tqdm.auto import tqdm
from lib.data import build_prefill_string
from lib.intervention import load_model, forward_pass_logits, extract_logit_stats

In [13]:
# Load model
model, tokenizer = load_model(MODEL_NAME, use_nnsight=True)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded Qwen/Qwen3-4B with nnsight
Loaded Qwen/Qwen3-4B with nnsight


In [14]:
# Load COT results (from notebook 02)
cots_path = RESULTS_DIR / "cots.jsonl"
assert cots_path.exists(), f"Run 02_generate_cots.ipynb first. Missing: {cots_path}"

cots = []
with open(cots_path) as f:
    for line in f:
        entry = json.loads(line)
        if entry["error"] is None and entry["cot_text"] is not None:
            cots.append(entry)

print(f"Loaded {len(cots)} valid COT results")

Loaded 128 valid COT results


In [15]:
def parse_condition(condition_name: str) -> int | None:
    """Parse condition name to get the layer index for zeroing.
    Returns None for self_prefill (no intervention).
    """
    if condition_name == "self_prefill":
        return None
    elif condition_name.startswith("zeroed_layer_"):
        return int(condition_name.split("_")[-1])
    else:
        raise ValueError(f"Unknown condition: {condition_name}")


def run_condition(condition_name: str, cots: list[dict]):
    """Run a single experimental condition across all COT results."""
    zero_at_layer = parse_condition(condition_name)
    
    # Resume logic
    done_ids = {
        int(p.stem.split("_", 1)[-1])  # parse "{condition}_{problem_id}.json"
        for p in INTERVENTION_CACHE.glob(f"{condition_name}_*.json")
    }
    todo = [c for c in cots if c["problem_id"] not in done_ids]
    print(f"\n[{condition_name}] Resuming: {len(done_ids)} done, {len(todo)} remaining")
    
    if not todo:
        return
    
    correct = 0
    total = 0
    
    for entry in tqdm(todo, desc=condition_name):
        try:
            prefill = build_prefill_string(entry["question"], entry["cot_text"])
            logits = forward_pass_logits(prefill, zero_at_layer=zero_at_layer)
            stats = extract_logit_stats(logits, entry["gold_answer"])
            
            cache_entry = {
                "problem_id": entry["problem_id"],
                "question": entry["question"],
                "gold_answer": entry["gold_answer"],
                "condition": condition_name,
                "cot_text": entry["cot_text"],
                "predicted_answer": stats["predicted_answer"],
                "top1_token": stats["top1_token"],
                "top1_prob": stats["top1_prob"],
                "gold_token_rank": stats["gold_token_rank"],
                "logits_top10": stats["logits_top10"],
                "error": None,
            }
            
            if stats["predicted_answer"] == entry["gold_answer"]:
                correct += 1
            total += 1
            
        except Exception:
            cache_entry = {
                "problem_id": entry["problem_id"],
                "question": entry["question"],
                "gold_answer": entry["gold_answer"],
                "condition": condition_name,
                "cot_text": entry["cot_text"],
                "predicted_answer": None,
                "top1_token": None,
                "top1_prob": None,
                "gold_token_rank": None,
                "logits_top10": None,
                "error": traceback.format_exc(),
            }
        
        fname = f"{condition_name}_{entry['problem_id']}.json"
        (INTERVENTION_CACHE / fname).write_text(json.dumps(cache_entry))
    
    if total > 0:
        print(f"[{condition_name}] Batch accuracy: {correct}/{total} = {correct/total:.1%}")

In [16]:
# Run the selected condition(s)
if CONDITION == "all":
    conditions_to_run = CONDITIONS
else:
    conditions_to_run = [CONDITION]

print(f"Conditions to run: {conditions_to_run}")

for cond in conditions_to_run:
    run_condition(cond, cots)

Conditions to run: ['self_prefill', 'zeroed_layer_0', 'zeroed_layer_9', 'zeroed_layer_18', 'zeroed_layer_27', 'zeroed_layer_35']


ValueError: invalid literal for int() with base 10: 'prefill_127'

In [8]:
# Aggregate results per condition into results/*.jsonl
for cond in CONDITIONS:
    entries = []
    for p in sorted(INTERVENTION_CACHE.glob(f"{cond}_*.json"), key=lambda x: int(x.stem.split("_")[-1])):
        entries.append(json.loads(p.read_text()))
    
    if not entries:
        continue
    
    output_path = RESULTS_DIR / f"{cond}.jsonl"
    with open(output_path, "w") as f:
        for entry in entries:
            f.write(json.dumps(entry) + "\n")
    
    valid = [e for e in entries if e["error"] is None]
    correct = sum(1 for e in valid if e["predicted_answer"] == e["gold_answer"])
    errors = len(entries) - len(valid)
    
    print(f"{cond}: {correct}/{len(valid)} correct ({correct/len(valid):.1%}), {errors} errors -> {output_path}")

self_prefill: 0/2 correct (0.0%), 126 errors -> /workspace/10-4-2026/results/self_prefill.jsonl
zeroed_layer_0: 0/1 correct (0.0%), 127 errors -> /workspace/10-4-2026/results/zeroed_layer_0.jsonl


ZeroDivisionError: division by zero

In [ ]:
# Quick summary of all conditions
print("\n" + "="*60)
print("SUMMARY")
print("="*60)

# Include normal (from COTs)
cots_valid = [c for c in cots if c["error"] is None]
normal_correct = sum(1 for c in cots_valid if c["predicted_answer"] == c["gold_answer"])
print(f"{'normal':<25} {normal_correct}/{len(cots_valid)} = {normal_correct/len(cots_valid):.1%}")

for cond in CONDITIONS:
    result_path = RESULTS_DIR / f"{cond}.jsonl"
    if not result_path.exists():
        print(f"{cond:<25} not yet run")
        continue
    
    entries = [json.loads(l) for l in result_path.read_text().splitlines()]
    valid = [e for e in entries if e["error"] is None]
    correct = sum(1 for e in valid if e["predicted_answer"] == e["gold_answer"])
    print(f"{cond:<25} {correct}/{len(valid)} = {correct/len(valid):.1%}")

In [9]:
# Diagnostic: inspect error messages from cached files
import json
from pathlib import Path

error_files = list(INTERVENTION_CACHE.glob("*.json"))
print(f"Total cached intervention files: {len(error_files)}")

# Show first few errors
shown = 0
for p in sorted(error_files)[:5]:
    entry = json.loads(p.read_text())
    if entry.get("error"):
        print(f"\n--- {p.name} ---")
        # Show last 5 lines of traceback
        lines = entry["error"].strip().split("\n")
        for line in lines[-5:]:
            print(line)
        shown += 1
        if shown >= 3:
            break

if shown == 0:
    print("No errors found in cache files.")

Total cached intervention files: 768

--- self_prefill_1.json ---
    result = forward_call(*args, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/qwen3/modeling_qwen3.py", line 61, in forward
    hidden_states = hidden_states.to(torch.float32)

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 139.72 GiB of which 19.12 MiB is free. Process 3477400 has 127.37 GiB memory in use. Process 3480831 has 12.31 GiB memory in use. Of the allocated memory 11.52 GiB is allocated by PyTorch, and 135.99 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

--- self_prefill_10.json ---
    result = forward_call(*args, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/qwen